# 10 — MLP model (cluster ready)

Evaluate A2 deep MLP against matched logistic regression using frozen development data only.

Use a dedicated `.venv-mlp` on the Linux GPU cluster; do not replace the existing classifier or agent environment. See `cluster/MLP_README.md` for installation, upload and Slurm commands. No local training is required.

Required inputs: `data/processed/d1_train.csv` and **`data/processed/model_cv_assignments.csv`** from notebook 02. Outer folds are restored by job ID, never regenerated. Optional historical comparison files are listed in the cluster guide.

The notebook defaults to 5 repeats × 5 outer folds. The Slurm launcher defaults to one diagnostic outer fold using the same training settings; use `MLP_RUN_MODE=full` for all 25. Both modes require a visible GPU. Completed model/fold predictions resume, but interrupted in-progress folds restart. Metrics and provenance are exported and the cluster launcher saves the executed notebook even after a cell error. No test data is loaded.


In [ ]:
from pathlib import Path
import hashlib
import sys
import os
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "src/d1_a2_mlp.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the complete repository folder, not a standalone notebook copy.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import d1_a2_mlp as experiment

DATA_FILE = ROOT / "data/processed/d1_train.csv"
if not DATA_FILE.is_file():
    raise FileNotFoundError("Missing data/processed/d1_train.csv from the completed notebook 00. Restore the agreed frozen file in this repository before continuing.")
train = experiment.load_development(DATA_FILE)
data_sha = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
RECORDED_DATA_SHA = "61117d0309733e4dcf28d1f9836588ffaf72b7b44479113a742533134cad57ce"
if data_sha != RECORDED_DATA_SHA:
    raise ValueError("Frozen training file differs from the agreed dataset. Restore it before training.")
print(f"Development pool: {len(train):,} postings | {int(train.fraudulent.sum()):,} frauds | {train.group_id.nunique():,} groups")
print("SHA256:", data_sha)
print("Matches recorded data:", data_sha == RECORDED_DATA_SHA)
print("Reserved final test not loaded.")

## 1. Candidate model

Keep A2 inputs and the team's saved outer folds, with a freshly trained matched logistic-regression baseline. Replace the classifier with a **512 → 256 → 128 → 64 → sigmoid MLP**, ReLU, dropout **0.4 / 0.4 / 0.3 / 0.3**, and per-fit balanced class weights.

This is a candidate experiment. Historical five-fold numbers below are teammate-reported context, not results from the requested 25-fold run. Final-test performance remains unknown.


## 2. Training choices and tuning actually performed

| Component | Current choice | How it was chosen |
|---|---|---|
| Inputs | A2, up to 30,000 word/bigram features | Carried forward from the team |
| Network and dropout | Four hidden layers above | Fixed candidate; no successful architecture search established |
| Optimizer | Adam 0.001; batch 64 | Retained setting; 0.001 is the only MLP learning-rate candidate in this grouped run |
| Class weights | Balanced, recomputed per fit | Fixed imbalance treatment |
| Training duration | Up to 50 epochs, patience 5 | Lowest loss on separate stopping groups inside training; then refit for that fixed duration |
| Classification threshold | One cutoff per outer fold | Highest F1 on training-side inner out-of-fold predictions |
| LR baseline | C = 0.1, 0.3 or 1.0 | Highest mean inner AP |

These are the actual selection steps—not a claim that the MLP learning rate or architecture was exhaustively fine-tuned. Earlier random-row tuning is not evidence of grouped performance. Model settings, checkpoint and threshold never use outer scoring labels.

In [ ]:
# Match 02: 5 repeats × 5 outer folds. Completed reference folds resume.
# To limit a diagnostic run, set N_REPEATS = 1 and OUTER_LIMIT = 1.
N_REPEATS = int(os.environ.get("MLP_REPEATS", "5"))
outer_limit = os.environ.get("MLP_OUTER_LIMIT", "all")
OUTER_LIMIT = None if outer_limit == "all" else int(outer_limit)
CONFIG = experiment.GroupedConfig(
    repeats=N_REPEATS, outer_limit=OUTER_LIMIT,
    outer_splits=5, inner_splits=3, checkpoint_splits=5,
    tfidf_max_features=30000, dense_weight=1.0,
    lr_Cs=(0.1, 0.3, 1.0), mlp_rates=(0.001,),
    max_epochs=50, patience=5, batch_size=64, seed=42)

import tensorflow as tf
tf.config.threading.set_intra_op_parallelism_threads(int(os.environ.get("SLURM_CPUS_PER_TASK", "4")))
tf.config.threading.set_inter_op_parallelism_threads(1)
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("No TensorFlow GPU detected. Check Slurm allocation, NVIDIA driver and CUDA dependencies before training.")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print("TensorFlow:", tf.__version__, "| GPUs:", gpus)
print("Environment versions:", experiment._versions())


## 3. Run the grouped comparison

Restore the required saved outer folds. Inner tuning and MLP stopping folds are generated inside training only, using the pinned environment. Completed outer model/fold predictions are cached atomically. A default full run involves approximately 200 neural training phases and 250 LR fits; measure cluster timing on a diagnostic fold first. No epoch-level resume or deployable-model export is provided.


In [ ]:
team_assignments = ROOT / "data/processed/model_cv_assignments.csv"
FOLDS = experiment.split_plan(train, CONFIG, team_assignments=team_assignments)
result = experiment.run_comparison(
    train, CONFIG, ROOT / "data/processed/09_a2_mlp", folds=FOLDS)
print(f"Completed {result['completed_outer_folds']}/{result['total_outer_folds']} configured outer folds.")

## 4. Export metrics and tuning choices

Report mean outer-fold metrics, paired corrected intervals, and combined decisions separately per repeat. Saved 02 logistic-regression and revised-03 LightGBM results are included only when their outer-fold identities match; LightGBM also requires its provenance plan. Historical training protocols may differ, so these rows are references rather than controlled classifier-only ablations. Missing or incompatible results are explicitly listed. Saved models are not retrained.


In [ ]:
from src.d1_model_metrics import export_comparison
REPORT_DIR = ROOT / "reports/tables/09_mlp" / Path(result["cache_dir"]).name / os.environ.get("SLURM_JOB_ID", "interactive")
metrics = export_comparison(ROOT, train, CONFIG, result, output_dir=REPORT_DIR)
print("Reports saved:", metrics["output_dir"])
print("Mean outer-fold comparison:")
display(metrics["model_comparison"][["model", "evaluated_folds", "expected_folds", "complete",
    "tfidf_max_features", "pr_auc", "precision", "recall", "f1", "false_positive_rate"]].round(4))
print("Combined decisions per repeat:")
display(metrics["combined_decisions_by_repeat"].round(4))
print("MLP epoch, settings and threshold selected inside training:")
display(metrics["mlp_tuning_choices"].round(4))
print("Paired MLP differences with corrected intervals (diagnostic runs may have too few folds):")
display(metrics["paired_mlp_vs_others"].round(4))
excluded = metrics["comparison_sources"].query("status == 'excluded'")
if not excluded.empty:
    print("Historical results excluded from this comparison:")
    display(excluded[["model", "reason"]])
print("Reserved final test not accessed.")

## Historical context and outputs

The teammate-reported one-repeat/five-fold checkpoint had MLP mean AP/F1 **0.6776 / 0.6188**, versus A2 LR **0.5914 / 0.5151**. These are not verified results of this notebook's 25-fold configuration. Use the exported matched comparison after completing the run; do not promote a model based on a diagnostic fold.

Fold prediction caches: `data/processed/09_a2_mlp/<cache hash>/`.
Metric CSVs and manifest: `reports/tables/09_mlp/<cache hash>/<job ID>/`.
Executed cluster notebook and environment snapshot: `reports/09_mlp_runs/<job ID>/`.
Only predictions are cached; a selected MLP would still need a separate final-development fit and model export for the agent.
